# Distance between two distributions (and information)

_Metrics & Evaluation — measuring models, data & statistics_

**How different are these two histograms? Many ways to put one number on it.**

You have two distributions — two histograms, two bags of numbers. Maybe last month's
       traffic vs this month's. Maybe real images vs ones your model dreamed up. The whole question of
       this lesson is one sentence: "how different are these two histograms?"

       A distribution is just how often each value shows up — the shape of a histogram once you
       turn the bar heights into fractions that add up to 1. Compare two of them and you get a single
       number: 0 means identical, bigger means more different.

---

This notebook builds those one-number comparisons from scratch, one family at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scipy + numpy

There are several ways to put one number on how different two distributions are. We build them in four steps: (1) information-theory quantities (entropy, cross-entropy, KL, JS), (2) perplexity and mutual information, (3) sample-based distances (Wasserstein, energy, total variation, Hellinger), and (4) MMD with a Gaussian kernel.

### Step 1 — Information theory: entropy, cross-entropy, KL, JS

Start from two histograms `P` and `Q` over the same bins. Entropy `H(P)` measures `P`'s own uncertainty. Cross-entropy is `H(P) + KL(P||Q)`. KL divergence `KL(P||Q)` measures the extra cost of coding `P` with `Q`'s code — and it is *asymmetric*, so `KL(P||Q)` differs from `KL(Q||P)`. JS divergence symmetrizes KL via the midpoint distribution; note `jensenshannon` returns the *distance*, so we square it to get the divergence.

In [ ]:
import numpy as np
from scipy.stats import entropy, wasserstein_distance, energy_distance
from scipy.spatial.distance import jensenshannon

# Two histograms P and Q over the SAME bins (each sums to 1).
P = np.array([0.5, 0.3, 0.2])
Q = np.array([0.2, 0.3, 0.5])

H_P = entropy(P, base=2)                            # entropy of P, in bits
H_PQ = entropy(P, base=2) + entropy(P, Q, base=2)  # cross-entropy = H(P) + KL(P||Q)
KL = entropy(P, Q, base=2)                         # KL(P || Q): second arg -> divergence
KL_rev = entropy(Q, P, base=2)                     # KL(Q || P): NOT equal in general
JS = jensenshannon(P, Q, base=2) ** 2              # JS DIVERGENCE (function returns the DISTANCE = sqrt)

print("H(P)=%.3f  CE=%.3f  KL=%.3f  KL_rev=%.3f  JS=%.3f" % (H_P, H_PQ, KL, KL_rev, JS))

### Step 2 — Perplexity and mutual information

Perplexity is `2` raised to the entropy — the "effective number of choices" a model faces. Mutual information measures how much two variables share: it is the KL divergence between their joint distribution and the product of their marginals (zero when independent). Pointwise mutual information (PMI) is the same comparison for a single cell, telling you whether one specific pair co-occurs more or less than chance.

In [ ]:
# Perplexity from entropy: 2 ** H, the "effective number of choices".
ppl = 2 ** entropy(P, base=2)
print("perplexity =", round(ppl, 3))

# Mutual information from a joint table p(x,y): I = KL(joint || product-of-marginals).
joint = np.array([[0.30, 0.10],
                  [0.10, 0.50]])
px = joint.sum(1, keepdims=True)         # row marginals p(x)
py = joint.sum(0, keepdims=True)         # column marginals p(y)

MI = np.sum(joint * np.log2(joint / (px * py)))        # mutual information, bits
PMI = np.log2(joint[1, 1] / (px[1, 0] * py[0, 1]))     # pointwise MI of one cell
print("MI=%.3f bits  PMI(1,1)=%.3f bits" % (MI, PMI))

### Step 3 — Sample-based distances

These distances work directly on raw samples, no binning required. Wasserstein (Earth Mover's) distance measures how far mass must move to turn one sample into the other; energy distance is a related sample comparison. Total variation is half the L1 gap between two histograms, and Hellinger distance compares their square roots — both bounded in [0, 1].

In [ ]:
# Wasserstein and energy distance work on raw SAMPLES.
a = np.array([2.9, 3.1, 3.4, 3.0, 2.7])   # sample from group A
b = np.array([1.9, 2.1, 1.7, 2.3, 2.0])   # sample from group B
print("Wasserstein =", round(wasserstein_distance(a, b), 3))
print("Energy      =", round(energy_distance(a, b), 3))

# Total variation & Hellinger on the two histograms P, Q:
TV = 0.5 * np.sum(np.abs(P - Q))                            # half the L1 gap
Hel = np.sqrt(0.5 * np.sum((np.sqrt(P) - np.sqrt(Q)) ** 2)) # Hellinger distance
print("TV=%.3f  Hellinger=%.3f" % (TV, Hel))

### Step 4 — Maximum Mean Discrepancy (MMD)

MMD compares two samples by mapping them through a kernel and measuring the gap between their mean embeddings. With a Gaussian (RBF) kernel, MMD² combines the average within-sample similarity of each group minus twice the cross-group similarity — zero when the samples look identical, larger as they diverge. We set the kernel bandwidth `sigma` from the pooled spread of both samples.

In [ ]:
# MMD from scratch with a Gaussian (RBF) kernel, on the raw samples a, b.
def mmd2(x, y, sigma):
    x = x[:, None]
    y = y[:, None]
    rbf = lambda u, v: np.exp(-(u - v.T) ** 2 / (2 * sigma ** 2))   # RBF similarity
    m, n = len(x), len(y)

    xx = rbf(x, x).sum() / m**2          # average within-x similarity
    yy = rbf(y, y).sum() / n**2          # average within-y similarity
    xy = rbf(x, y).sum() / (m * n)       # average cross similarity
    return xx + yy - 2 * xy              # MMD^2

sigma = np.std(np.concatenate([a, b]))
print("MMD^2 =", round(mmd2(a, b, sigma), 4))

## Visualize the data & results

_Take one real feature from load_wine, build a histogram for two wine classes, and ask: do KL, JS, Hellinger, and Wasserstein all flag the same shift — at different scales?_

### Step 1 — Build two class-conditional histograms

We pick the `flavanoids` feature from `load_wine` and split it by wine class, giving two real samples. Binning both over the *same* 8 bins turns them into comparable histograms. A small smoothing constant `eps` keeps every bin nonzero so KL never divides by zero, then we renormalize so each histogram sums to 1.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from scipy.stats import entropy, wasserstein_distance
from scipy.spatial.distance import jensenshannon

d = load_wine()                              # 178 real wines, 13 chemical features
X, y = d.data, d.target
fi = list(d.feature_names).index("flavanoids")
a = X[y == 0, fi]                            # class 0 flavanoids (59 wines)
b = X[y == 1, fi]                            # class 1 flavanoids (71 wines)

# Two class-conditional histograms over the SAME 8 bins.
lo = min(a.min(), b.min())
hi = max(a.max(), b.max())
bins = np.linspace(lo, hi, 9)

pa = np.histogram(a, bins=bins)[0]
pa = pa / pa.sum()
pb = np.histogram(b, bins=bins)[0]
pb = pb / pb.sum()

eps = 1e-3                                    # smoothing so KL never divides by zero
P = pa + eps
P /= P.sum()
Q = pb + eps
Q /= Q.sum()

### Step 2 — Compare the same shift across five metrics

Now we score the gap between the two classes five ways. The two KL directions differ (asymmetry again); JS gives a symmetric distance; Hellinger compares square roots; and Wasserstein works on the raw samples. They all flag the same real shift, but on different scales — a reminder that the *choice* of metric matters as much as the number.

In [ ]:
kl_fwd = entropy(P, Q, base=2)               # KL(c0 || c1)
kl_rev = entropy(Q, P, base=2)               # KL(c1 || c0)  -> different! (asymmetry)
js = jensenshannon(pa, pb, base=2)           # JS DISTANCE (sqrt of JS divergence)
hell = np.sqrt(0.5 * np.sum((np.sqrt(P) - np.sqrt(Q)) ** 2))   # Hellinger
wass = wasserstein_distance(a, b)            # earth-mover, on raw samples

print("KL fwd=%.3f  KL rev=%.3f  JS=%.3f  Hellinger=%.3f  Wasserstein=%.3f"
      % (kl_fwd, kl_rev, js, hell, wass))
# -> 1.250        2.542        0.687     0.608          0.934

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A monitoring job computes $D_{\mathrm{KL}}(P_{\text{today}}\,\|\,Q_{\text{train}})$ on a categorical feature and suddenly returns inf. The feature looks fine. What happened and how do you fix it without throwing away the metric?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look for a category that appears today but had zero count in the training reference $Q$. — _KL has a term $p_i\log(p_i/q_i)$; if $q_i=0$ while $p_i\gt 0$ you divide by zero inside the log, giving $+\infty$._
- Add a small smoothing constant to every bin of both histograms before normalizing (e.g. add $10^{-3}$ to each count). — _Additive (Laplace) smoothing removes the hard zero so the log is finite, while barely changing well-populated bins._
- Or switch the drift score to JS divergence or Wasserstein. — _JS uses the midpoint $M=\tfrac12(P+Q)$, which is never zero where either is positive, so it stays finite by construction; Wasserstein has no division at all._

**Answer:** A new category with zero reference probability made a $q_i=0$, so KL hit $+\infty$. Fix it by additive smoothing (a tiny constant in every bin) or by switching to JS divergence / Wasserstein, both of which stay finite even with disjoint support. This is KL's classic zero-support pitfall.

</details>

**Problem 2.** Two distributions sit on completely separate ranges: $P$ is a spike at $0$, $Q$ is a spike at $5$. As you slide $Q$ from $5$ down toward $1$, what do KL/JS report versus Wasserstein, and which is more useful for training a generator?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that as long as the spikes don't overlap, $P$ and $Q$ share no support. — _KL is infinite (zero in one where the other is positive); JS is pinned at its maximum (1 bit) and flat — it can't tell 5 from 2._
- Compute Wasserstein: it is just the distance the mass must move, $|0-5|=5$, then $4$, then $1$ as $Q$ slides in. — _Earth-mover distance measures how far mass is, not whether bins coincide, so it changes smoothly even with disjoint support._
- Pick the metric whose value (and gradient) changes as the spikes approach. — _A generator needs a signal that says "you're getting warmer"; a flat JS gives no gradient, Wasserstein does._

**Answer:** KL is $\infty$ and JS is stuck at its max (1 bit) the whole time the supports are disjoint — both are flat and useless for moving $Q$ closer. Wasserstein reports $5\to4\to1$, decreasing smoothly, giving a usable gradient. This is exactly why WGAN (Wasserstein GAN) replaced the JS-based objective.

</details>

**Problem 3.** In NLP you want to know whether the bigram "machine learning" is a real collocation or just two common words landing next to each other. Which information measure answers this, and how do you read its sign?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use pointwise mutual information: $\mathrm{PMI}(x,y)=\log\frac{p(x,y)}{p(x)p(y)}$ for $x=$"machine", $y=$"learning". — _PMI compares the observed co-occurrence $p(x,y)$ to what you'd expect if the words were independent, $p(x)p(y)$._
- Estimate $p(x,y)$ from how often the pair occurs, and $p(x),p(y)$ from each word's own frequency. — _These are just counts divided by totals over the corpus._
- Read the sign: PMI $\gt 0$ means they co-occur more than chance (a collocation); $\approx 0$ means independent; $\lt 0$ means they avoid each other. — _The log is positive when the ratio exceeds 1, i.e. the pair is over-represented._

**Answer:** Pointwise mutual information (PMI). A large positive PMI says "machine" and "learning" appear together far more than their individual frequencies predict — a genuine collocation. PMI near 0 means independent; negative means they repel. Averaging PMI over all word pairs gives the total mutual information $I(X;Y)$.

</details>